Dựa trên bài lab đã học ở tuần trước (CNN Thuần), hãy thay đổi thành các tập dữ liệu khác như:
- Cat and dog
- CIFAR-10
- PlantVillage
Sử dụng các phương pháp phân tích dữ liệu, cân bằng dữ liệu,...
Cố gắng thay đổi các tham số sao cho độ chính xác phải lớn hơn 90% và tránh trường hợp overfitting.

Không sử dụng các mô hình re-train hoặc các mô hình như Resnet, Convnext tiny,....
Deadline: 21/03/2026

1. CIFAR-10 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
import os

# ==================== PHÂN TÍCH DỮ LIỆU ====================
transform_basic = transforms.Compose([transforms.ToTensor()])
trainset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_basic)
classes = trainset_raw.classes

# Kiểm tra phân bố lớp (CIFAR-10 cân bằng)
labels = np.array(trainset_raw.targets)
unique, counts = np.unique(labels, return_counts=True)
print("Phân bố lớp:", dict(zip(classes, counts)))

# Visualize 10 mẫu
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    img, label = trainset_raw[i]
    axes[i//5, i%5].imshow(img.permute(1,2,0))
    axes[i//5, i%5].set_title(classes[label])
    axes[i//5, i%5].axis('off')
plt.show()

# ==================== DATA AUGMENTATION & TRANSFORMS ====================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Weighted sampler (dù cân bằng nhưng vẫn dùng để ổn định)
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
sample_weights = [class_weights[label] for label in labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

trainloader = DataLoader(trainset, batch_size=128, sampler=sampler, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# ==================== MODEL CNN THUẦN (5 blocks) ====================
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.3),
            
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.4),
            
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.5),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 2 * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = CIFAR_CNN().cuda()
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

# ==================== TRAINING VỚI EARLY STOPPING ====================
best_acc = 0
patience = 8
counter = 0
epochs = 50

for epoch in range(epochs):
    model.train()
    for inputs, labels in trainloader:
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.cuda(), labels.cuda()
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    scheduler.step(acc)
    
    print(f'Epoch {epoch+1}/{epochs} - Accuracy: {acc:.2f}%')
    
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_cifar_cnn.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

print(f"Độ chính xác tốt nhất: {best_acc:.2f}%")

 49%|████▉     | 84.1M/170M [05:47<09:48, 147kB/s] 

2. Cats vs Dogs

In [ ]:
# Transform + augmentation mạnh hơn
transform_train = transforms.Compose([
    transforms.Resize(128),
    transforms.RandomResizedCrop(112),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

trainset = torchvision.datasets.ImageFolder('train', transform=transform_train)
# ... tiếp tục tương tự CIFAR nhưng chỉ 2 classes

# Model: giảm channel xuống còn 32-64-128-256 (vì binary)
# Accuracy dễ đạt 95-97% sau 20 epochs.

3. PlantVillage (38 classes)